# Field - JavaScript

All 8 JavaScript examples from [docs/core/field.md](https://platob.github.io/yggdryl/core/field/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install @yggdryl/node
```

In [ ]:
const assert = require('node:assert/strict')
const { DataType, Field } = require('@yggdryl/node')

const field = new Field('price', 'decimal(18, 6)', false)

assert.equal(field.name, 'price')
assert.ok(field.dataType.equals(DataType.from('decimal(18, 6)')))
assert.equal(field.nullable, false)
assert.equal(field.size, 0)

assert.ok(Field.from(field.toString()).equals(field))
assert.ok(Field.from('price decimal(18, 6) NOT NULL').equals(field))

## A non-null struct field is the schema

In [ ]:
const assert = require('node:assert/strict')
const { DataType, Field, fields } = require('@yggdryl/node')

const schema = new Field(
  'trade',
  DataType.fromFields([
    fields.int64('id', { nullable: false }),
    fields.utf8('symbol', { nullable: true }),
  ]),
  false,
)

const children = schema.dataType
assert.equal(children.length, 2)
assert.equal(children.at(0).nullable, false)
assert.equal(children.getByName('symbol').name, 'symbol')
assert.deepEqual(children.keys(), ['id', 'symbol'])

## Metadata is a mapping

In [ ]:
const assert = require('node:assert/strict')
const { Field } = require('@yggdryl/node')

const field = new Field('price', 'float64', false, { venue: 'XPAR' })
field.set('currency', 'EUR')
field.update(new Map([['source', 'exchange']]))

assert.equal(field.size, 3)
assert.equal(field.has('venue'), true)
assert.equal(field.get('venue'), 'XPAR')
assert.equal(field.get('missing'), null)
assert.deepEqual([...field], [
  ['currency', 'EUR'],
  ['source', 'exchange'],
  ['venue', 'XPAR'],
])

assert.equal(field.delete('venue'), true)
assert.deepEqual(field.keys(), ['currency', 'source'])

## Reserved keys and protocol properties

In [ ]:
const assert = require('node:assert/strict')
const { Field, MimeType } = require('@yggdryl/node')

const field = new Field('payload', 'binary', false)

field.setParquetFieldId(17)
field.set('field:init', 'false')
field.setContentType('application/json; charset=utf-8')
field.setProperty('postgres', 'type', 'jsonb')

assert.equal(field.parquetFieldId, 17)
assert.equal(field.get('PARQUET:field_id'), '17')
assert.equal(field.get('field:init'), 'false')

assert.ok(field.mimeType.equals(MimeType.JSON))
assert.equal(field.getProperty('https', 'Content-Type'), field.contentType)
assert.equal(field.get('http:content-type'), field.contentType)
assert.deepEqual(field.propertyIter('postgres'), [{ key: 'type', value: 'jsonb' }])

## One protocol at a time

In [ ]:
const assert = require('node:assert/strict')
const { Field } = require('@yggdryl/node')

const field = new Field('price', 'int64', false)

field.iceberg.set('doc', 'closing price')
field.iceberg.update({ 'schema-id': '3', 'field-id': '7' })
field.postgres.set('type', 'numeric')

assert.equal(field.iceberg.get('doc'), 'closing price')
assert.equal(field.iceberg.key('doc'), 'iceberg:doc')
assert.equal(field.iceberg.size, 3)
assert.equal(field.mysql.size, 0)

// It is a view of the one metadata map, not a copy of part of it.
assert.equal(field.get('iceberg:doc'), 'closing price')
assert.equal(field.size, 4)
assert.deepEqual([...field.iceberg].sort(), [['doc', 'closing price'], ['field-id', '7'], ['schema-id', '3']])

assert.equal(field.iceberg.delete('field-id'), true)
assert.equal(field.iceberg.has('field-id'), false)
assert.equal(field.protocol('postgres').get('type'), 'numeric')

## A field can be a partition column

In [ ]:
const assert = require('node:assert/strict')
const { DataType, Field } = require('@yggdryl/node')

const schema = new Field(
  'row',
  DataType.fromFields([
    new Field('year', 'int32', false),
    new Field('venue', 'string', false),
    new Field('price', 'int64', false),
  ]),
  false,
).withPartitionFields(['year', 'venue'])

assert.equal(schema.hasPartitionFields, true)
assert.deepEqual(schema.partitionFieldNames(), ['year', 'venue'])
assert.equal(schema.dataType.getByName('year').isPartition, true)
assert.equal(schema.dataType.getByName('price').isPartition, false)

assert.equal(schema.withoutPartitionFields().dataType.length, 1)
assert.equal(schema.onlyPartitionFields().dataType.length, 2)

## Typed field aliases

In [ ]:
const assert = require('node:assert/strict')
const { Field, fields } = require('@yggdryl/node')

const id = fields.int64('id')
const symbol = fields.utf8('symbol', { nullable: true, metadata: { source: 'feed' } })
const at = fields.timestamp('at', 'us')

assert.ok(id instanceof Field)
assert.equal(id.dataType.toString(), 'int64')
assert.equal(symbol.get('source'), 'feed')
assert.equal(at.dataType.toString(), 'timestamp(us)')

## Comparing two fields

In [ ]:
const assert = require('node:assert/strict')
const { Field } = require('@yggdryl/node')

const left = new Field('price', 'float64', false, { venue: 'XPAR' })
const right = new Field('price', 'float64', true, { venue: 'XNAS' })

assert.equal(left.equals(right), false)
assert.deepEqual([...left.showDiffs(right)], [
  '≠ $.nullable: false → true',
  '≠ $.metadata["venue"]: "XPAR" → "XNAS"',
])
assert.equal(left.showDiff(left), '✓ equal')
assert.equal(left.showDiff(left, true, false), '')